# Analyzing Thematic Alignment in EMLP Conference Papers

This notebook analyzes the thematic alignment between papers published at EMLP(Empirical Methods in Natural Language Processing) and the conference's stated Aims&Scope, using sentence embedding and cosine similarity.

## Pipeline
1. Data collection via Semantic Scholar API
2. Text embedding using allenai-specter
3. Alignement score computation via cosine similarity
4. Results analysis and visualization

## 1. Installation
Install the required libraries:

In [ ]:
!pip install requests pandas sentence-transformers matplotlib scikit-learn

## 2. Data Collection
We collect abstacts from EMLP papers using the Semantic Scholar API, filtering only papers published at the "Conference on Empirical Methods in Natural Language Processing" with a non-empty abstract.

In [ ]:
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os
import json


# Skip if data already exists
if os.path.exists("../data/abstracts.json"):
    print("Data already collected, skipping...")
else:
    API_KEY = "your_API_key"

    BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search"
    PARAMS = {
    "query": "EMNLP empirical methods natural language processing",
    "fields": "title,abstract,year,venue",
    "limit": 100
    }

    HEADERS = {"x-api-key": API_KEY}


    articoli_validi = []
    offset = 0

    while len(articoli_validi) < 1000:
        PARAMS["offset"] = offset
    
        response = requests.get(BASE_URL, headers=HEADERS, params=PARAMS)   #Chiamata all'API
        data = response.json()                                              #Trasforma quello che riceve in un file json
        if response.status_code == 429:
            print("Troppe richieste, aspetto 10 secondi...")
            time.sleep(10)
            continue
    
        if "data" not in data or len(data["data"]) == 0:                    #La risposta non contiene il campo API o pagina vuota
            break
    
        for articolo in data["data"]:
            venue = articolo.get("venue", "")                               #prendo da che rivista deriva l'articolo, se non c'è restituisco una stringa vuota
            abstract = articolo.get("abstract", None)                       #prendo il contenuto dell'articolo, se non c'è restituisco None
        
            if venue == "Conference on Empirical Methods in Natural Language Processing" and abstract is not None:                   #controllo che la rivista sia quella giusta e che l'articolo abbia un contenuto
                articoli_validi.append({                                    #aggiugo l'articolo alla lista di articoli validi
                    "title": articolo.get("title", ""),
                    "abstract": abstract,
                    "year": articolo.get("year", ""),
                    "venue": venue
                })
    
        offset += 100                                                          #aggiorno l'ofsett per prendere i nuovi articoli
        time.sleep(3)                                                          #aspetto un secondo per evitare di sovraccaricare l'API
        print(f"Offset: {offset} | Articoli validi finora: {len(articoli_validi)}")

    print(f"Articoli raccolti: {len(articoli_validi)}")                        #numero di articoli ottenuti

    with open("data/abstracts.json", "w") as f:
        json.dump(articoli_validi, f, indent=2)                                #salvo i dati nel file json in modo leggibile

    print("Salvato in data/abstracts.json")

## 3. Embedding generation
We use allenai-specter, a BERT-based model pre-trained on scientific papers, to generate dense vector representations of each abstract and the Aim&Scopes text. 

In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Carica il modello
model = SentenceTransformer('allenai-specter')

# Carica l'Aims & Scope
with open("data/aims_and_scope.txt", "r") as f:
    aims_and_scope = f.read()

# Carica gli abstract
with open("data/abstracts.json", "r") as f:
    articoli = json.load(f)

# Embedding dell'Aims & Scope
print("Generando embedding Aims & Scope...")
embedding_aims = model.encode(aims_and_scope)

# Embedding degli abstract
print("Generando embedding abstract...")
abstracts_testi = [articolo["abstract"] for articolo in articoli]
embeddings_abstract = model.encode(abstracts_testi, show_progress_bar=True)

# Salva gli embedding
np.save("data/embedding_aims.npy", embedding_aims)
np.save("data/embeddings_abstract.npy", embeddings_abstract)

print("Embedding salvati!")

from sklearn.metrics.pairwise import cosine_similarity

# Calcola alignment scores
scores = cosine_similarity([embedding_aims], embeddings_abstract)[0]

# Aggiungi gli scores agli articoli
for i, articolo in enumerate(articoli):
    articolo["alignment_score"] = float(scores[i])

# Salva i risultati
with open("results/results.json", "w") as f:
    json.dump(articoli, f, indent=2)

print("Alignment scores salvati in results/results.json")

## 4. Alignement analysis
We compute cosine similarity between each abstract and the Aim&Scope embedding as alignement score.

In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt

# Carica i risultati
with open("results/results.json", "r") as f:
    articoli = json.load(f)

#filtro gli articoli per il periodo 2019-2025
articoli = [a for a in articoli if a["year"] and 2019 <= a["year"] <= 2025]
scores = np.array([a["alignment_score"] for a in articoli])
anni = [a["year"] for a in articoli]
titoli = [a["title"] for a in articoli]

print(f"Articoli nel periodo 2019-2025: {len(articoli)}")

# Estrai gli scores
scores = np.array([a["alignment_score"] for a in articoli])
anni = [a["year"] for a in articoli]
titoli = [a["title"] for a in articoli]

# Istogramma degli alignment scores
plt.figure(figsize=(10, 6))
plt.hist(scores, bins=30, color='steelblue', edgecolor='black')
plt.axvline(scores.mean(), color='red', linestyle='--', label=f'Media: {scores.mean():.3f}')
plt.xlabel('Alignment Score')
plt.ylabel('Numero di articoli')
plt.title('Distribuzione degli Alignment Scores - EMNLP')
plt.legend()
plt.tight_layout()
plt.savefig("results/histogram.png")
plt.show()
print("Istogramma salvato!")



# Drift temporale
from collections import defaultdict

scores_per_anno = defaultdict(list)
for a in articoli:
    if a["year"]:
        scores_per_anno[a["year"]].append(a["alignment_score"])

anni_ordinati = sorted(scores_per_anno.keys())
medie_annuali = [np.mean(scores_per_anno[a]) for a in anni_ordinati]

plt.figure(figsize=(12, 6))
plt.plot(anni_ordinati, medie_annuali, marker='o', color='steelblue')
plt.xlabel('Anno')
plt.ylabel('Alignment Score medio')
plt.title('Drift Temporale degli Alignment Scores - EMNLP')
plt.grid(True)
plt.tight_layout()
plt.savefig("results/drift_temporale.png")
plt.show()
print("Grafico drift temporale salvato!")



# Numero di articoli per anno
conteggio_per_anno = [len(scores_per_anno[a]) for a in anni_ordinati]

fig, ax1 = plt.subplots(figsize=(12, 6))

# Linea degli scores medi
ax1.plot(anni_ordinati, medie_annuali, marker='o', color='steelblue', label='Score medio')
ax1.set_xlabel('Anno')
ax1.set_ylabel('Alignment Score medio', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Barre del conteggio articoli su asse secondario
ax2 = ax1.twinx()
ax2.bar(anni_ordinati, conteggio_per_anno, alpha=0.3, color='orange', label='Numero articoli')
ax2.set_ylabel('Numero di articoli', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

plt.title('Drift Temporale e Numero di Articoli per Anno - EMNLP')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.savefig("results/drift_temporale_contestualizzato.png")
plt.show()
print("Grafico contestualizzato salvato!")



# Outlier
soglia_bassa = scores.mean() - 2*scores.std()
soglia_alta = scores.mean() + 2*scores.std()

outlier_bassi = sorted(articoli, key=lambda x: x["alignment_score"])[:5]
outlier_alti = sorted(articoli, key=lambda x: x["alignment_score"], reverse=True)[:5]

print("\n--- 5 ARTICOLI MENO ALLINEATI ---")
for a in outlier_bassi:
    print(f"Score: {a['alignment_score']:.4f} | Anno: {a['year']} | {a['title'][:60]}")

print("\n--- 5 ARTICOLI PIÙ ALLINEATI ---")
for a in outlier_alti:
    print(f"Score: {a['alignment_score']:.4f} | Anno: {a['year']} | {a['title'][:60]}")